# 04 · Análise dos resultados

Lê os arquivos da avaliação e monta a comparação dos 3 sistemas: a tabela com todas as métricas, o gráfico de acerto × custo, os erros por pergunta, o roteamento, o verificador e a concordância entre o juiz, a revisão humana e a checagem automática. **Só lê arquivos**: roda no Colab com o runtime padrão (só CPU) ou localmente.

| Arquivo (em `resultados/`) | De onde vem |
|---|---|
| `execucoes.csv`, `execucoes.jsonl` | notebook 03 (as 60 execuções) |
| `juiz.csv`, `juiz.jsonl` | `avaliacao/juiz.py` (o juiz, com o Claude pelo Claude Code) |
| `revisao_manual.csv` | a revisão humana dos vereditos (o modelo é gerado na Parte 7) |
| `ambiente.json` | notebook 03 (hardware, modelo e versões) |

**Acerto** = veredito **correta**; parcial conta como erro. O acerto que vale (*acerto final*) é o da revisão humana quando a execução foi revisada, e o do juiz quando não foi. Os gráficos são gravados em `resultados/graficos/`, para o README e os slides.

In [ ]:
# @title Parte 1 · Carregar os resultados
import os, subprocess, sys

REPO_URL = "https://github.com/felipeafigueiredo/text-to-sql-and-cypher.git"
try:
    import google.colab                                         # no Colab: clona o repositório
    REPO_DIR = "/content/text-to-sql-and-cypher"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
except ImportError:                                             # local: o notebook está em notebooks/
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO_DIR)

RESULTADOS_DIR = os.path.join(REPO_DIR, "resultados")          # ou uma pasta do Drive com os mesmos arquivos
GRAFICOS_DIR = os.path.join(RESULTADOS_DIR, "graficos")

import pandas as pd
from IPython.display import Markdown, display
from avaliacao import analise as an
pd.set_option("display.max_colwidth", 120)

DADOS = an.carregar(RESULTADOS_DIR)
DF = an.por_execucao(DADOS)
print(f"{len(DF)} execuções · {DF['pergunta_id'].nunique()} perguntas · sistemas: {', '.join(DF['sistema'].unique().astype(str))}")
print(f"juiz: {'sim' if DADOS['juiz'] is not None else 'ainda não'} · "
      f"revisão humana: {0 if DADOS['revisao'] is None else len(DADOS['revisao'])} execuções")

---
## Parte 2 · Ambiente

O hardware, o modelo e as versões da rodada das 60 execuções (gravados pelo notebook 03).

In [ ]:
# @title Ambiente da rodada
amb = DADOS["ambiente"]
if amb:
    libs = amb.pop("bibliotecas", {})
    display(pd.Series({**amb, **{f"lib {k}": v for k, v in libs.items()}}, name="valor").to_frame())
else:
    print("ambiente.json não encontrado")

---
## Parte 3 · Tabela comparativa

Uma coluna por sistema, com as métricas obrigatórias (acerto, trajetória, latência média/mediana/p95, tokens de entrada e saída somados entre os agentes, chamadas ao modelo e às ferramentas, CPU, RAM e GPU) e as do tema: **consultas válidas** (consultas SQL e Cypher escritas pelo modelo que rodaram sem erro) e tentativas até a primeira consulta válida.

In [ ]:
# @title Tabela comparativa
TABELA = an.tabela_comparativa(DF)
TABELA.to_csv(os.path.join(RESULTADOS_DIR, "tabela_comparativa.csv"))
display(an.formatar(TABELA))

---
## Parte 4 · Acerto × custo

Cada sistema é um ponto. O custo aparece em dois painéis com a mesma escala de acerto (tokens por pergunta e latência mediana), e não num gráfico de dois eixos, porque as duas medidas têm escalas diferentes. Abaixo, a distribuição da latência de cada execução.

In [ ]:
# @title Acerto × custo e latência
an.grafico_acerto_custo(DF, GRAFICOS_DIR);
an.grafico_latencia(DF, GRAFICOS_DIR);

---
## Parte 5 · Erros por pergunta

Onde cada sistema errou: o veredito de cada execução, o acerto por categoria de pergunta e por banco esperado, e a lista das execuções que não foram consideradas corretas, com a justificativa do juiz.

In [ ]:
# @title Veredito por pergunta
an.grafico_mapa(DF, "veredito_juiz", GRAFICOS_DIR);
if DF["veredito_humano"].notna().any():
    an.grafico_mapa(DF, "veredito_humano", GRAFICOS_DIR, "vereditos_por_pergunta_humano.png");

In [ ]:
# @title Acerto por categoria e por banco
an.grafico_acerto_por(DF, "categoria", an.CATEGORIAS, "Acerto por categoria de pergunta", "acerto_por_categoria.png", GRAFICOS_DIR);
an.grafico_acerto_por(DF, "banco", an.BANCOS, "Acerto por banco esperado", "acerto_por_banco.png", GRAFICOS_DIR);
display(an.pct(an.acerto_por(DF, "categoria", an.CATEGORIAS)))
display(an.pct(an.acerto_por(DF, "banco", an.BANCOS)))

In [ ]:
# @title Execuções que não foram consideradas corretas
erros = DF[DF["acerto_final"] == False][["execucao", "categoria", "banco", "veredito_juiz", "veredito_humano",
                                          "checagem_falhas", "justificativa_juiz"]]
print(f"{len(erros)} de {len(DF)} execuções")
display(erros.set_index("execucao"))

---
## Parte 6 · Roteamento e verificador

**Sistema 2:** a rota escolhida pelo roteador contra o banco esperado da pergunta. Nas perguntas P11 e P12, que o SQL também responde, a rota `sql` conta como certa (T46).

**Sistema 3:** as respostas que o verificador devolveu, com o veredito do juiz para a primeira resposta do agente (antes) e para a final (depois).

In [ ]:
# @title Roteamento (sistema 2)
acerto_rota, matriz_rota = an.roteamento(DF)
if acerto_rota is not None:
    print("acerto de rota por banco esperado:")
    display(an.pct(acerto_rota.rename("rota certa")))
    print("rota esperada × rota escolhida:")
    display(matriz_rota)

In [ ]:
# @title Verificador (sistema 3)
dev = DF[DF["sistema"] == "verificador"]["devolucoes"].fillna(0)
print(f"devoluções: {int(dev.sum())} em {int((dev > 0).sum())} das {len(dev)} execuções do sistema 3")
mud = an.verificador(DADOS)
if mud is not None and len(mud):
    display(mud.set_index("execucao"))
    display(mud["efeito"].value_counts().to_frame("execuções"))

---
## Parte 7 · Revisão manual e concordância

A especificação pede a verificação manual dos vereditos. A célula abaixo cria `resultados/revisao_manual.csv` (se ainda não existir), com a pergunta, o gabarito, a resposta, a checagem automática e o veredito do juiz em cada linha. O grupo preenche a coluna **`veredito_humano`** com `correta`, `parcial` ou `incorreta` (e `observacao`, se quiser), sem olhar o veredito do juiz antes de decidir, se possível. Pode ser aberto no Excel ou no Google Planilhas: o separador `;` também é aceito. Depois, rode o notebook de novo.

Concordância em **acerto** (certo/errado) entre os três: humano, juiz e checagem automática, com o **kappa de Cohen** (concordância além do acaso: 1 é perfeita, 0 é a do acaso).

In [ ]:
# @title Modelo da revisão manual
caminho, criado = an.modelo_revisao(RESULTADOS_DIR, DADOS)
print(("criado: " if criado else "já existe (não foi sobrescrito): ") + caminho)

In [ ]:
# @title Concordância
pares, matriz = an.concordancia(DF)
display(pares.assign(concordância=pares["concordância"].map(lambda v: f"{v:.0%}"), kappa=pares["kappa"].round(2)))
if matriz is not None:
    print("vereditos: juiz (linhas) × humano (colunas)")
    display(matriz)
    print("execuções em que juiz e humano discordam no acerto:")
    display(an.discordancias(DF).set_index("execucao"))
else:
    print("ainda sem revisão humana: preencha veredito_humano em revisao_manual.csv e rode de novo")

In [ ]:
# @title Custo do juiz
c = an.custo_do_juiz(DADOS)
if c is not None:
    display(c.to_frame("juiz"))

---
## Parte 8 · Conclusão

Os números que a conclusão precisa, gerados dos resultados (para conferir: a conclusão em texto é escrita pelo grupo, a partir deles e da análise dos erros acima).

In [ ]:
# @title Fatos para a conclusão
for f in an.fatos(DF, DADOS):
    print("•", f)

### Conclusão

Rodada única: 20 perguntas × 3 sistemas, `qwen3:8b` (Q4_K_M) no Ollama, numa T4 do Colab. Acerto pela revisão humana, que concordou com o juiz em 98% das execuções.

**1. O verificador acertou mais e custou mais.** Acerto de 70%, contra 55% do agente único e 40% do roteamento. O preço é o dobro de tokens (+110%), uma latência mediana 41% maior (9,8 s × 7,0 s) e uma cauda bem mais longa (p95 de 72 s × 33 s). No gráfico de acerto × custo não há sistema que domine os outros: o verificador compra acerto com tokens e tempo, e o roteamento economiza tokens (−27%) mas perde acerto.

**2. A hipótese do roteamento não se confirmou.** O roteador acertou o banco nas 20 perguntas, então o problema não foi a classificação. Com menos contexto, os especialistas escreveram mais consultas que rodam (93% válidas, contra 88% do agente único), mas não as que respondem à pergunta. O sistema errou 5 das 6 perguntas de Cypher e as 3 híbridas, quase sempre na parte do especialista do grafo, e o padrão se repete:
- **desiste no primeiro vazio**: escreve uma consulta errada (com seta no `BORDERS`, procurando "Seazone" no nome do anúncio), recebe zero linhas e responde que não há resultado;
- **nunca chamou o `buscar_entidade`**, que estava entre as ferramentas dele, e inventou um id na P10 (`listing_id = 123`);
- **lê mal o próprio resultado**: na P15 e na P16, a consulta devolveu o próprio bairro e ele concluiu que o bairro "não tem vizinhos".

O agente único tinha as mesmas ferramentas de grafo e acertou P09, P11 e P16. A leitura mais provável é que o esquema do SQL e as regras gerais, que o especialista não vê, também orientavam as consultas de grafo (por exemplo, a busca do anfitrião pelo nome antes de consultar). Separar o contexto tirou informação útil, não só ruído.

**3. A hipótese do verificador se confirmou, com um limite claro.** Ele devolveu 6 respostas (9 devoluções): 4 viraram certas (P06, P09, P12, P17), 1 passou de incorreta a parcial (P15) e 1 continuou errada (P10). Nenhuma resposta certa foi estragada, então o "na dúvida, aprove" e a exigência de evidência funcionaram. O limite: o verificador é o mesmo modelo de 8B e cai nas mesmas armadilhas que o agente. Ele aprovou o "742 avaliações em 2025" da P04, que é o contador da vida toda (o caso aparece como exemplo no prompt dele), as consultas com seta no `BORDERS` da P13 e da P14, e o "209 anúncios" da P18. Um verificador pega o erro que ele mesmo reconhece; um erro de premissa que os dois compartilham passa.

**4. Onde o modelo teve mais dificuldade.**
- **Cypher** (6 perguntas): 33% no agente único, 17% no roteamento e 50% no verificador, contra 70%, 60% e 80% no SQL. A causa principal foi a **direção do `BORDERS`**: cada par de vizinhos está gravado numa direção só, e o modelo escreve `-[:BORDERS]->`. O servidor devolve um aviso explicando isso, mas o modelo lê o aviso e responde com o resultado errado assim mesmo. Para um modelo deste tamanho, o aviso precisaria ser um erro, ou a relação precisaria existir nas duas direções.
- **Híbridas** (3): 33%, 0% e 67%. Exigem levar os nomes de um banco para o outro sem perder nenhum.
- **Armadilhas dos contadores**: na P04 (avaliações de 2025) os três sistemas usaram o `number_of_reviews`, que conta a vida toda; na capciosa P18, os três deram só os 209 anúncios em São Paulo, sem o total do Airbnb inteiro que a busca mostrava ao lado.
- **Respostas vazias**: na P17, os sistemas 1 e 2 terminaram com a resposta em branco depois de gastar a geração.
- **Casos especiais bem resolvidos**: item inexistente (P19) e fora do escopo (P20) estão certos nos três sistemas.

**5. O juiz é confiável para esta tarefa.** O juiz (Claude Sonnet com acesso somente leitura ao servidor MCP) concordou com a revisão humana em 98% das execuções (kappa 0,97), e os vereditos só diferiram em 3 casos, todos "parcial" do juiz. Ele é um pouco mais rígido: exigiu os dois números na P12 quando a pergunta só pedia quem tinha mais. A checagem automática por expressões regulares, sem modelo, concordou com o humano em 97% (kappa 0,93); ela falha quando a resposta traz uma afirmação extra errada (S2-P02, com anúncios de outros bairros), que o juiz pega. O juiz custou o equivalente a US$ 0,55 pelos 66 julgamentos. A ordem dos sistemas é a mesma pelo humano, pelo juiz e pela checagem, então a conclusão não depende de quem julgou.

**Limitações.** É uma rodada só, com temperatura 0: as diferenças entre sistemas são de 3 a 6 perguntas, e cada pergunta vale 5 pontos percentuais. Há um modelo só, de 8B; os resultados dizem mais sobre modelos pequenos do que sobre as arquiteturas em geral. Parte das melhorias do servidor (T47, T54, T55) veio de falhas vistas em perguntas da avaliação durante o desenvolvimento. Elas são gerais (não entregam a consulta de nenhuma pergunta) e valem igual para os três sistemas, mas o resultado absoluto pode estar um pouco otimista.

**Mais de um agente se justifica?** Para este problema e este modelo, **sim, no formato do verificador, e não no do roteamento**. O verificador acertou 3 perguntas a mais que o agente único (70% × 55%) sem estragar nenhuma resposta certa. Isso vale o dobro de tokens quando o erro custa caro, o que é o caso de uma consulta que devolve um número errado com cara de certo. O roteamento custa menos (−27% de tokens), mas acertou 3 perguntas a menos: dividir o contexto por banco piorou o modelo pequeno em vez de focá-lo. Os erros que sobram nos três sistemas são de premissa (a direção da relação, o contador errado), que nenhuma das duas arquiteturas corrige sozinha e que pedem mudanças nos dados ou no servidor.